This notebook measures the response times of our pipeline. By changing the pipeline we used this notebook to observe the new response times

In [ ]:
import sys
import os
import time
import random
from tqdm import tqdm
import numpy as np

# Add root folder into paths
sys.path.append(os.path.abspath('..'))

# Now we can import our own classes/functions
from utils.modeling import make_LLM_pipeline, generate_response
from utils.datahandling import load_database

In [ ]:
LLM_model_path = "../saved_models/meltemi 2025-07-08 10:16:23"
instructions_path = '../instructions/v1 LLM instructions.txt'

# Read the instructions that will be given to the model
with open(instructions_path, 'r', encoding='utf-8') as file:
    system_instructions = file.read()

# Load data
data = load_database('../datasets/rephrased_faq_v1.json')
data.extend(load_database('../datasets/rephrased_faq_v2.json'))
data.extend(load_database('../datasets/rephrased_faq_v3.json'))

In [ ]:
# Load LLM
generator = make_LLM_pipeline(LLM_model_path)

In [ ]:
# Check response times for 100 requests
ms_per_token = []
texts = []
for i in tqdm(range(100)):
    prompt = random.choice(data)['input']
    start = time.perf_counter()
    text = generate_response(prompt, system_instructions, generator.tokenizer, generator.model)
    texts.append(text)
    elapsed = (time.perf_counter() - start) * 1000  # ms
    ms_per_token.append(elapsed / len(text))


print('ms per token stats:')
print('average:', np.mean(ms_per_token))
print('min', np.min(ms_per_token))
print('percentile 10', np.percentile(ms_per_token, 10))
print('percentile 25', np.percentile(ms_per_token, 25))
print('percentile 50', np.percentile(ms_per_token, 50))
print('percentile 75', np.percentile(ms_per_token, 75))
print('percentile 90', np.percentile(ms_per_token, 90))
print('max', np.max(ms_per_token))

In [ ]:
# spreadsheet friendly print
print(np.mean(ms_per_token), '\t', np.min(ms_per_token), '\t',\
      np.percentile(ms_per_token, 10), '\t', np.percentile(ms_per_token, 25), '\t',\
        np.percentile(ms_per_token, 50), '\t', np.percentile(ms_per_token, 75), '\t',\
            np.percentile(ms_per_token, 90), '\t', np.max(ms_per_token))